# MNIST MLP3: every-step TraceLogRG, 50 epochs

Maximum-intervention test of the existing one-sided trace-log correction:

- identical AdamW baseline and AdamW + TraceLogRG runs;
- correction checked on every minibatch step;
- full projection strength and no norm cap;
- 50 epochs;
- WeightWatcher `alpha` and `ERG_gap` read directly from `analyze(..., ERG=True)`;
- no beta-E or spectral-shape controller.

The extra loss and milestone diagnostics distinguish **slower convergence** from a possible reduction in **late overfitting**.

In [ ]:
from pathlib import Path
import math, sys
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, torch, weightwatcher as ww
from IPython.display import display

ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "rg_trace_log").is_dir():
        ROOT = p; break
    q = p / "optimizers" / "trace_log_tracker"
    if (q / "rg_trace_log").is_dir():
        ROOT = q; break
if ROOT is None:
    raise RuntimeError("Open this notebook from a clone of rg_optimizers.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_trace_log import MNISTExperimentConfig, run_mnist_comparison

RUNS = ["AdamW baseline", "AdamW + TraceLogRG"]
LAYERS = ["fc1", "fc2", "fc3"]
RUN_COLOR = {"AdamW baseline": "#2563A6", "AdamW + TraceLogRG": "#238B57"}
BLUE = {"fc1": "#9ECAE1", "fc2": "#4292C6", "fc3": "#08519C"}
GREEN = {"fc1": "#A1D99B", "fc2": "#41AB5D", "fc3": "#006D2C"}
MARKER = {"fc1": "o", "fc2": "s", "fc3": "^"}

config = MNISTExperimentConfig(
    seed=1337, epochs=50, batch_size=128, learning_rate=1e-3,
    weight_decay=1e-2, grad_clip_norm=1.0,
    rg_mode="one_sided", rg_normalization="weightwatcher",
    rg_gamma=0.10, rg_ridge_relative=1e-6, rg_min_retained=5,
    rg_correction_scale=1.0, rg_max_correction_ratio=None,
    rg_apply_every_steps=1, rg_warmup_steps=0,
    ww_min_evals=10, ww_max_evals=None, n_log_shells=5,
    min_retained_for_beta=20, min_decades_for_beta=0.50,
    train_eval_max_batches=None,
)
assert config.epochs == 50 and config.rg_apply_every_steps == 1
assert config.rg_correction_scale == 1.0 and config.rg_max_correction_ratio is None

print("Root:", ROOT)
print("Torch:", torch.__version__)
print("WeightWatcher:", getattr(ww, "__version__", "unknown"))
print(config)

result = run_mnist_comparison(config, data_dir=ROOT / "data", progress=True)
OUTPUT = ROOT / "results_every_step_all_layers_50_epochs"
result.save(OUTPUT)
print("Saved:", OUTPUT.resolve())

In [ ]:
# WeightWatcher metrics and performance plots.
need = {
    "run", "epoch", "layer_name", "status", "alpha", "ERG_gap",
    "alpha_source", "ERG_gap_source",
}
missing = need - set(result.weightwatcher.columns)
if missing:
    raise RuntimeError(f"Missing columns {sorted(missing)}; pull the latest main branch.")

w = result.weightwatcher.copy()
w["layer"] = w.layer_name.astype(str).str.split(".").str[-1]
w = w.loc[
    w.status.eq("ok") & w.run.isin(RUNS) & w.layer.isin(LAYERS)
].copy()
w["alpha"] = pd.to_numeric(w.alpha, errors="coerce")
w["ERG_gap"] = pd.to_numeric(w.ERG_gap, errors="coerce")
if w[["alpha", "ERG_gap"]].isna().any().any():
    raise RuntimeError("WeightWatcher returned missing alpha or ERG_gap.")
if not w.alpha_source.astype(str).eq("WeightWatcher").all():
    raise RuntimeError("alpha was not sourced directly from WeightWatcher.")
if not w.ERG_gap_source.astype(str).eq("WeightWatcher").all():
    raise RuntimeError("ERG_gap was not sourced directly from WeightWatcher.")

def finish(ax, title, ylabel, ref=None):
    if ref is not None:
        ax.axhline(ref, color="black", ls="--", lw=1.4, label=f"reference = {ref:g}")
    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.show()

def runplot(metric, title, ylabel):
    fig, ax = plt.subplots(figsize=(9.5, 5.2), dpi=135)
    for run in RUNS:
        g = result.performance.loc[result.performance.run.eq(run)].sort_values("epoch")
        ax.plot(g.epoch, g[metric], marker="o", ms=4, lw=2.4,
                color=RUN_COLOR[run], label=run)
    finish(ax, title, ylabel)

def layerplot(metric, title, ylabel, ref):
    d = w.loc[w.epoch.ge(1)]
    fig, ax = plt.subplots(figsize=(11.5, 6.2), dpi=135)
    for run in RUNS:
        palette = BLUE if run == "AdamW baseline" else GREEN
        for layer in LAYERS:
            g = d.loc[d.run.eq(run) & d.layer.eq(layer)].sort_values("epoch")
            ax.plot(g.epoch, g[metric], marker=MARKER[layer], ms=4, lw=2.1,
                    color=palette[layer], label=f"{run} — {layer.upper()}")
    finish(ax, title, ylabel, ref)
    for layer in LAYERS:
        fig, ax = plt.subplots(figsize=(9.5, 5.2), dpi=135)
        for run in RUNS:
            g = d.loc[d.run.eq(run) & d.layer.eq(layer)].sort_values("epoch")
            ax.plot(g.epoch, g[metric], marker="o", ms=4, lw=2.4,
                    color=RUN_COLOR[run], label=run)
        finish(ax, f"{title}: {layer.upper()}", ylabel, ref)

runplot("train_acc", "Training accuracy", "Training accuracy")
runplot("test_acc", "Test accuracy", "Test accuracy")
runplot("train_loss", "Training cross-entropy loss", "Training loss")
runplot("test_loss", "Test cross-entropy loss", "Test loss")
layerplot("alpha", r"WeightWatcher $\alpha$", r"WeightWatcher $\alpha$", 2.0)
layerplot("ERG_gap", "WeightWatcher ERG gap", "WeightWatcher ERG gap", 0.0)
display(w[["run", "epoch", "layer", "alpha", "ERG_gap"]].tail(18))

In [ ]:
# Correction size and firing frequency by layer and epoch.
s = result.rg_steps.copy()
if s.empty:
    raise RuntimeError("No step-level correction records.")
s["layer"] = (
    s.parameter.astype(str).str.replace(".weight", "", regex=False)
    .str.split(".").str[-1]
)
s = s.loc[s.layer.isin(LAYERS)].copy()
for col in ["correction_ratio", "base_trace_log_drift", "corrected_trace_log_drift"]:
    s[col] = pd.to_numeric(s[col], errors="coerce")
s["fired"] = s.status.eq("ok")
s["ratio_all"] = s.correction_ratio.fillna(0.0)
s["ratio_fired"] = s.correction_ratio.where(s.fired)
s["base_abs"] = s.base_trace_log_drift.abs().where(s.fired)
s["corrected_abs"] = s.corrected_trace_log_drift.abs().where(s.fired)

expected = math.ceil(60_000 / config.batch_size)
c = s.groupby(["epoch", "layer"], as_index=False).agg(
    steps=("global_step", "nunique"),
    opportunities=("global_step", "size"),
    fired=("fired", "sum"),
    fired_fraction=("fired", "mean"),
    failures=("status", lambda x: x.eq("geometry_failed").sum()),
    mean_ratio_all_steps=("ratio_all", "mean"),
    mean_ratio_when_fired=("ratio_fired", "mean"),
    median_ratio_when_fired=("ratio_fired", "median"),
    max_ratio=("ratio_all", "max"),
    base_abs=("base_abs", "sum"),
    corrected_abs=("corrected_abs", "sum"),
)
c["coverage"] = c.steps / expected
c["drift_residual"] = np.where(c.base_abs > 0, c.corrected_abs / c.base_abs, np.nan)
display(c)

RATIO = r"$\|\Delta W_{\rm correction}\|_F/\|\Delta W_{\rm AdamW}\|_F$"
def correction_plot(col, title, ylabel, ylim=None):
    fig, ax = plt.subplots(figsize=(10.5, 5.2), dpi=135)
    for layer in LAYERS:
        g = c.loc[c.layer.eq(layer)].sort_values("epoch")
        ax.plot(g.epoch, g[col], marker=MARKER[layer], ms=4, lw=2.4,
                color=GREEN[layer], label=layer.upper())
    if ylim is not None:
        ax.set_ylim(*ylim)
    finish(ax, title, ylabel)

correction_plot("mean_ratio_all_steps", "Mean correction per layer and epoch",
                f"Mean {RATIO} over all steps")
correction_plot("mean_ratio_when_fired", "Mean correction when the trigger fires",
                f"Mean {RATIO} over corrected steps")
correction_plot("max_ratio", "Largest correction per layer and epoch",
                f"Maximum {RATIO}")
correction_plot("fired_fraction", "Fraction of steps corrected per layer",
                "Corrected-step fraction", (-.02, 1.02))

heat = c.pivot(index="layer", columns="epoch",
               values="mean_ratio_all_steps").reindex(LAYERS)
fig, ax = plt.subplots(figsize=(15, 3.8), dpi=135)
im = ax.imshow(heat.to_numpy(float), aspect="auto", cmap="Greens")
ax.set_yticks(range(3), [x.upper() for x in LAYERS])
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set(xlabel="Epoch", ylabel="Layer",
       title="Mean correction / AdamW-step norm")
fig.colorbar(im, ax=ax, label=f"Mean {RATIO}")
plt.tight_layout(); plt.show()

In [ ]:
# Convergence versus late-overfitting diagnostics.
p = result.performance.loc[result.performance.epoch.ge(1)].copy()

def first_epoch(g, metric, threshold):
    x = g.loc[g[metric].ge(threshold), "epoch"]
    return int(x.iloc[0]) if len(x) else np.nan

rows = []
for run in RUNS:
    g = p.loc[p.run.eq(run)].sort_values("epoch")
    final = g.iloc[-1]
    best_acc = g.loc[g.test_acc.idxmax()]
    best_loss = g.loc[g.test_loss.idxmin()]
    late = g.tail(10)
    rows.append({
        "run": run,
        "epoch_train_acc_0.98": first_epoch(g, "train_acc", .98),
        "epoch_train_acc_0.99": first_epoch(g, "train_acc", .99),
        "epoch_train_acc_0.995": first_epoch(g, "train_acc", .995),
        "final_train_acc": final.train_acc,
        "final_test_acc": final.test_acc,
        "peak_test_acc": best_acc.test_acc,
        "peak_test_acc_epoch": int(best_acc.epoch),
        "peak_to_final_test_acc_drop": best_acc.test_acc - final.test_acc,
        "minimum_test_loss": best_loss.test_loss,
        "minimum_test_loss_epoch": int(best_loss.epoch),
        "final_test_loss": final.test_loss,
        "test_loss_rebound": final.test_loss - best_loss.test_loss,
        "mean_test_loss_last_10": late.test_loss.mean(),
        "mean_test_acc_last_10": late.test_acc.mean(),
        "final_loss_gap": final.test_loss - final.train_loss,
        "final_accuracy_gap": final.train_acc - final.test_acc,
    })
summary = pd.DataFrame(rows).set_index("run").reindex(RUNS)
display(summary.T)

base, rg = summary.loc[RUNS[0]], summary.loc[RUNS[1]]
delay = rg["epoch_train_acc_0.99"] - base["epoch_train_acc_0.99"]
late_loss_gain = base.mean_test_loss_last_10 - rg.mean_test_loss_last_10
rebound_gain = base.test_loss_rebound - rg.test_loss_rebound
acc_drop_gain = base.peak_to_final_test_acc_drop - rg.peak_to_final_test_acc_drop

print("Epoch delay to 99% train accuracy (RG - baseline):", delay)
print("Late-10 test-loss advantage (baseline - RG):", late_loss_gain)
print("Test-loss rebound reduction (baseline - RG):", rebound_gain)
print("Peak-to-final accuracy-drop reduction (baseline - RG):", acc_drop_gain)
print()
if pd.notna(delay) and delay > 1 and max(late_loss_gain, rebound_gain) <= 1e-3 and acc_drop_gain <= 1e-4:
    print("VERDICT: primarily slower convergence; no matching late generalization benefit.")
elif max(late_loss_gain, rebound_gain) > 1e-3 or acc_drop_gain > 1e-4:
    print("VERDICT: evidence consistent with less late overfitting; repeat across seeds and then sweep cadence.")
else:
    print("VERDICT: no clear convergence or late-generalization advantage.")

# Generalization-gap plots.
p["loss_gap"] = p.test_loss - p.train_loss
p["accuracy_gap"] = p.train_acc - p.test_acc
for metric, title, ylabel in [
    ("loss_gap", "Loss generalization gap", "Test loss − train loss"),
    ("accuracy_gap", "Accuracy generalization gap", "Train accuracy − test accuracy"),
]:
    fig, ax = plt.subplots(figsize=(9.5, 5.2), dpi=135)
    for run in RUNS:
        g = p.loc[p.run.eq(run)].sort_values("epoch")
        ax.plot(g.epoch, g[metric], marker="o", ms=4, lw=2.4,
                color=RUN_COLOR[run], label=run)
    finish(ax, title, ylabel)

## How to interpret this run

Every-step projection is a **maximum-intervention diagnostic**, not the proposed final cadence.

- If training is slower but late test loss, test-loss rebound, and peak-to-final test accuracy are not better, the correction is merely obstructing convergence.
- If early training catches up and late test degradation is smaller, there is evidence that the direction is acting as an overfitting barrier.
- If WeightWatcher spectra change while predictive metrics do not, the effect is spectral but observationally neutral in this regime.

Only after seeing a real late benefit should we sweep the cadence (for example, every 1, 5, 10, and 25 steps). No beta-E control is used here.